## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy import linalg as la

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True


## Utility functions

In [ ]:
def vcol(x):
    return x.reshape((x.size, 1))

def vrow(x):
    return x.reshape((1, x.size))

def load_project_data(path):
    data = []
    labels = []
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            data.append([float(x) for x in parts[:-1]])
            labels.append(int(parts[-1]))
    return np.array(data).T, np.array(labels)

def logpdf_GAU_ND(X, mu, C):
    M = X.shape[0]
    XC = X - mu
    sign, logdet = np.linalg.slogdet(C)
    invC = np.linalg.inv(C)
    const = -0.5 * M * np.log(2 * np.pi)
    return const - 0.5 * logdet - 0.5 * (XC * (invC @ XC)).sum(axis=0)

def compute_mu_C(D):
    mu = vcol(D.mean(axis=1))
    DC = D - mu
    C = (DC @ DC.T) / D.shape[1]
    return mu, C

def loglikelihood(X, mu, C):
    return logpdf_GAU_ND(X, mu, C).sum()


## Load the project dataset

The project dataset is the provided training set with 6 features and 2 classes. The labels are `0` and `1`, and samples are stored column-wise after loading.

In [ ]:
D, L = load_project_data('../../../Project/trainData.txt')
# D.shape, L.shape, np.unique(L), np.bincount(L)

## Split the data by class

For the qualitative project analysis, it is acceptable to use the whole training set, as explicitly allowed by the PDF. Therefore, the Gaussian parameters below are estimated using all available training samples for each class.

In [ ]:
D0 = D[:, L == 0]
D1 = D[:, L == 1]

# ['Feature 1', 'Feature 2', 'Feature 3', 'Feature 4', 'Feature 5', 'Feature 6']
feature_names = [f'Feature {i+1}' for i in range(D.shape[0])]


# D0.shape, D1.shape
# feature_names


## Fit a 1D Gaussian to each feature of each class

For every feature and each class separately, compute the ML estimate of the mean and variance. Since each model is uni-variate, the covariance matrix is a `1 x 1` matrix containing the feature variance.

In [ ]:
results = []
for cls, Dc in [(0, D0), (1, D1)]:
    for i in range(D.shape[0]):         # D.shape[0] --> Number of the Features
        Xi = Dc[i:i+1, :]               #  extracts one single feature (row i) from the class data matrix Dc
        mu_i, C_i = compute_mu_C(Xi)
        ll_i = loglikelihood(Xi, mu_i, C_i)
        results.append({
            'class': cls,
            'feature': i + 1,
            'mu': float(mu_i.ravel()[0]),
            'var': float(C_i.ravel()[0]),
            'std': float(np.sqrt(C_i.ravel()[0])),
            'loglikelihood': float(ll_i)
        })

# Shows the Result for the Class 0        
#results[:5]

# Shows the Result for the Class 1
# results[6:]



## Parameter table

The table below reports the estimated Gaussian parameters for every class-feature pair. These numbers are useful when commenting on the histogram shapes and on the relative spread of the two classes.

In [ ]:
import pandas as pd # type: ignore
params_df = pd.DataFrame(results).set_index(['class', 'feature'])
params_df


## Plot normalized histograms with Gaussian densities

Each subplot shows the normalized histogram of one feature for one class together with the fitted Gaussian density.

In [ ]:
def plot_feature_gaussians(Dc, cls_label, feature_names):
    fig, axs = plt.subplots(2, 3, figsize=(18, 12))  
    axs = axs.ravel()
    for i in range(Dc.shape[0]):
        Xi = Dc[i:i+1, :]
        mu_i, C_i = compute_mu_C(Xi)
        x_min, x_max = Xi.min(), Xi.max()
        margin = 0.2 * (x_max - x_min if x_max > x_min else 1.0)
        XPlot = np.linspace(x_min - margin, x_max + margin, 1000)
        YPlot = np.exp(logpdf_GAU_ND(vrow(XPlot), mu_i, C_i))

        axs[i].hist(Xi.ravel(), bins=40, density=True, alpha=0.6, color='steelblue', edgecolor='black')
        axs[i].plot(XPlot, YPlot, color='red', linewidth=2)
        axs[i].set_title(f'Class {cls_label} — {feature_names[i]}')
        axs[i].set_xlabel(feature_names[i])
        axs[i].set_ylabel('Density')
    plt.tight_layout()
    plt.show()

plot_feature_gaussians(D0, 0, feature_names)
plot_feature_gaussians(D1, 1, feature_names)


## Overlay both classes on the same feature

To compare the two classes more directly, the following plots show the histograms and Gaussian fits of class `0` and class `1` on the same axes for each feature. This makes it easier to comment on overlap, symmetry, multimodality, and mismatch with the Gaussian assumption.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10))
axs = axs.ravel()

for i in range(D.shape[0]):
    X0 = D0[i:i+1, :]
    X1 = D1[i:i+1, :]
    mu0, C0 = compute_mu_C(X0)
    mu1, C1 = compute_mu_C(X1)

    x_min = min(X0.min(), X1.min())
    x_max = max(X0.max(), X1.max())
    margin = 0.2 * (x_max - x_min if x_max > x_min else 1.0)
    XPlot = np.linspace(x_min - margin, x_max + margin, 1000)
    Y0 = np.exp(logpdf_GAU_ND(vrow(XPlot), mu0, C0))
    Y1 = np.exp(logpdf_GAU_ND(vrow(XPlot), mu1, C1))

    axs[i].hist(X0.ravel(), bins=35, density=True, alpha=0.45, label='Class 0', color='royalblue')
    axs[i].hist(X1.ravel(), bins=35, density=True, alpha=0.45, label='Class 1', color='orange')
    axs[i].plot(XPlot, Y0, color='blue', linewidth=2)
    axs[i].plot(XPlot, Y1, color='darkorange', linewidth=2)
    axs[i].set_title(feature_names[i])
    axs[i].legend()

plt.tight_layout()
plt.show()


### What the Crossing Points Mean


The two Gaussian curves cross at one or two points. At a crossing point:

$$
p(x \mid \text{class 0}) = p(x \mid \text{class 1})
$$

This means the two densities are **equal** at that x value. The LLR (log-likelihood ratio) at that point is:

$$
\text{LLR}(x) = \log \frac{p(x \mid \text{class 1})}{p(x \mid \text{class 0})} = 0
$$

So the crossing point is exactly the **decision boundary** — the threshold where you switch from predicting class 0 to predicting class 1.

***

### What Each Region Means

look at what happens in each region between and outside the crossing points:

- **Where class 1 curve is higher** than class 0 curve → the density of class 1 is larger → LLR > 0 → **predict class 1**

<br>


- **Where class 0 curve is higher** than class 1 curve → the density of class 0 is larger → LLR < 0 → **predict class 0**

<br>

- **At the crossing point** → both densities are equal → LLR = 0 → this is the **threshold**

<br>

The samples that fall **inside the overlap region** between the two bell curves are the ones that are hard to classify correctly. The classifier will inevitably make mistakes there, because both classes have similar density in that region.

***

### Why the Gaussian Fit is Bad for Features 5 and 6

As the professor explained earlier in the class, features 5 and 6 have **4 clusters** total — 2 clusters per class. This means the real distribution of each class is **not a single bell curve** — it is actually a **mixture** of two sub-groups pushed together.

When you force a single Gaussian to fit this multi-modal (multi-hump) data:

- Class 0 has **3 peaks** in the histogram → a single Gaussian cannot capture this shape
- Class 1 has **2 peaks** in the histogram → again a single Gaussian is a poor fit

So the yello Gaussian curve sits somewhere in the middle and **does not match the actual histogram bars well at all**. The professor pointed this out explicitly — it is a bad fit.

***



The professor looked at features 5 and 6 and explained that even though the Gaussian fit is clearly wrong (the bell curve does not match the multi-peak histogram), the classifier **can still do something useful**:

> *"In the region where there are many more class 1 points, the density for class 1 is larger than for class 0 — so the LLR is positive there. In the regions where class 0 dominates, the LLR tends to be negative."*

His point was: **a bad fit does not automatically mean a useless classifier.** Even with a wrong Gaussian shape, if the two fitted curves manage to roughly separate where the two classes are concentrated, the LLR will still point in the right direction for most samples. The classifier extracts some information even from an incorrect model.

He then said:

> *"You can see that if you remove features 5 and 6 from the classifier you will actually get worse results. So that means the models are able to extract information even if the features are modeled poorly."*

***

### What Happens With 2 Crossing Points

When both Gaussians have different means AND different variances (which is the case here for features 1, 2, 5 and 6), the two bell curves can cross **twice** instead of once. 

Now you have **three regions** instead of two:


|  region A  |      region B       |  region C  |
|------------|---------------------|------------|
| LLR < 0    |      LLR > 0        |  LLR < 0   |
| predict 0  |      predict 1      |  predict 0 |

<br>

***

### Why Features 1 \& 2 Have 2 Crossing Points

For features 1 and 2, the professor said the two classes have:

- **Almost the same mean** (the bell curves are centered at nearly the same point)
- **Different variances** (one class is more spread out than the other — one bell curve is wider and flatter, the other is taller and narrower)

When two Gaussians have the **same mean but different variances**, they cross **twice** — once on each side of the shared center:

The taller curve is higher in the **middle** (near the shared mean), and the wider curve is higher on the **tails** (far from the mean).

So the decision regions are:

- **Far left** (tail): wider class wins → predict that class
- **Middle** (near the mean): taller/narrower class wins → predict that class
- **Far right** (tail): wider class wins → predict that class again

<br>
<hr>

### What the Professor Said About Features 1 \& 2 Specifically

The professor said that for features 1 and 2, **even with 2 crossing points, the classifier is basically useless**. Here is why:

Because the means are almost identical, the two bell curves sit almost on top of each other. The crossing points are very close together — the "middle region" where the taller curve wins is tiny. The vast majority of samples from **both** classes fall in the overlapping zone.

He said:

> *"For those classes you would model them with a single Gaussian — the means are very similar. These will be the models. We have a small shift in the means, very small because the means are not exactly the same. But the variance will be the same. So everything here is the same to one class, everything on the other to the other class. So I make more or less 50% error for all those samples."*

In other words — the two fitted Gaussians look almost **identical**, so the LLR is close to 0 almost everywhere. The classifier cannot decide which class a sample belongs to because both densities are almost equal for every sample. The result is roughly **50% error** — no better than random guessing.

<br>
<hr>

## Side by Side Comparison

| Questions ? |  Features 1 \& 2 | Features 5 \& 6 |
| :-- | :-- | :-- |
| **Why 2 crossings?** | Same mean, different variance | Bad Gaussian fit on multi-cluster data |
| **Gaussian fit quality** | Reasonable fit, but classes overlap badly | Poor fit — data has multiple peaks |
| **LLR usefulness** | Near zero everywhere — almost useless | Still extracts some information |
| **Classification result** | ~50% error — near random | Better than random — still contributes |
| **Professor's conclusion** | Linear classifier won't work here | Even a bad model can do something useful |

**The key difference:** 

- features 1 \& 2 have 2 crossings because the classes look nearly identical (same mean).

-  Features 5 \& 6 have a complex crossing structure because the Gaussian model is simply a poor fit to multi-cluster data. 

- Both have 2 crossings, but for completely different geometric reasons.

## Project discussion and answers



### What do you observe?
By fitting a univariate Gaussian to each feature and each class, it becomes clear that the quality of the Gaussian approximation depends strongly on the feature. Some feature distributions are reasonably centered and bell-shaped, so the Gaussian curve follows the histogram fairly well. Other features show visible asymmetry, heavier tails, or multiple local peaks, and in those cases the Gaussian model is only a rough approximation rather than a precise fit.

### Are there features for which the Gaussian densities provide a good fit?
Yes. A Gaussian density provides a good qualitative fit when the histogram is approximately unimodal, fairly symmetric around the mean, and smoothly concentrated around a central region. In those cases, the fitted red curve tends to match both the center and the spread of the histogram reasonably well, which means that the ML-estimated mean and variance capture most of the visible structure of that one-dimensional feature.

### Are there features for which the Gaussian model seems significantly less accurate?
Yes. A Gaussian model becomes less accurate when the histogram is clearly skewed, has heavy tails, or appears multimodal. If a feature contains multiple peaks or separated clusters, a single Gaussian cannot represent that shape well, because one Gaussian can only model one symmetric bell-shaped mode with one mean and one variance.

### Why does the Gaussian fit work better for some features than for others?
The Gaussian model is a low-complexity parametric model, so it works best when the true distribution of the feature is close to a single bell-shaped distribution. When the feature distribution is generated by more complicated mechanisms, such as mixtures of subgroups, bounded effects, or asymmetric variation, the one-dimensional Gaussian is too simple and cannot adapt to the real shape of the histogram.

### What is the role of maximum-likelihood estimation here?
Maximum-likelihood estimation provides the Gaussian parameters that make the observed samples most probable under the chosen model. In the univariate case, this means estimating the sample mean and the sample variance for each feature of each class, then using those parameters to draw the fitted density curve on top of the histogram.

### Final conclusion
The project shows that the univariate Gaussian assumption is sometimes a reasonable approximation, but not universally accurate for every feature. Therefore, Gaussian densities can still be useful as simple probabilistic models, but the plots also suggest that some project features may require richer models, or at least careful interpretation, when later used for classification.
